In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import cv2

In [ ]:
import cv2
import os
import pandas as pd

def load_images_to_dataframe(image_dir):

    data = []

    for file_name in os.listdir(image_dir):
        file_path = os.path.join(image_dir, file_name)
        

        if file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):

            img = cv2.imread(file_path)
            

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            

            img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_AREA)
            

            alpha = 1.2  
            beta = 20 
            img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

            image_id = int(os.path.splitext(file_name)[0])
            

            data.append({
                'image_id': image_id,
                'image_data': img
            })


    df = pd.DataFrame(data)
    return df


In [ ]:
df_train = load_images_to_dataframe('/kaggle/input/tamillang/Misogyny Meme Detection/train/')
df_test = load_images_to_dataframe('/kaggle/input/tamillang/Misogyny Meme Detection/test')
df_dev = load_images_to_dataframe('/kaggle/input/tamillang/Misogyny Meme Detection/dev')

In [ ]:
train_csv = pd.read_csv('/kaggle/input/tamillang/Misogyny Meme Detection/train/train.csv')
dev_csv = pd.read_csv('/kaggle/input/tamillang/Misogyny Meme Detection/dev/dev.csv')
test_csv = pd.read_csv('/kaggle/input/tamillang/Misogyny Meme Detection/test/test.csv')

train = pd.merge(train_csv, df_train, on='image_id', how='inner')
dev = pd.merge(dev_csv, df_dev, on='image_id', how='inner')
test = pd.merge(test_csv, df_test, on='image_id', how='inner')


In [ ]:
pip install git+https://github.com/indic-transliteration/indic_transliteration_py/@master -U

In [ ]:
def text_preprocessing(text):
    import re
    pattern = re.compile('[@#\/]\S+')
    text = pattern.sub(r'',text)

    pattern = re.compile('\d+')
    text = pattern.sub(r'', text)

    pattern = re.compile(r'https?:\/\/\S+|www\.\S+|ftp:\/\/\S+|mailto:\S+|https?:')

    # First remove URLs
    text = pattern.sub('', text)

    # Remove newline characters (\n) and carriage returns (\r)
    text = text.replace('\n', ' ').replace('\r', '')

    # Remove extra spaces (including multiple spaces)
    text = re.sub(r'\s+', ' ', text).strip()

    import string
    punc = string.punctuation

    text = text.translate(str.maketrans('','',punc))

    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # Emoticons
        "\U0001F300-\U0001F5FF"  # Symbols & Pictographs
        "\U0001F680-\U0001F6FF"  # Transport & Map Symbols
        "\U0001F1E0-\U0001F1FF"  # Flags (iOS)
        "\U00002700-\U000027BF"  # Dingbats
        "\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        "\U00002600-\U000026FF"  # Miscellaneous Symbols
        "\U00002B50-\U00002B55"  # Stars and other symbols
        "]+",
        flags=re.UNICODE
    )

    text = emoji_pattern.sub(r'', text)

    tamil_stopwords = [
    "ஒரு", "என்று", "மற்றும்", "இந்த", "இது", "என்ற", "கொண்டு", "என்பது", "பல", "ஆகும்",
    "அல்லது", "அவர்", "நான்", "உள்ள", "அந்த", "இவர்", "என", "முதல்", "என்ன", "இருந்து",
    "சில", "என்", "போன்ற", "வேண்டும்", "வந்து", "இதன்", "அது", "அவன்", "தான்", "பலரும்",
    "என்னும்", "மேலும்", "பின்னர்", "கொண்ட", "இருக்கும்", "தனது", "உள்ளது", "போது", "என்றும்",
    "அதன்", "தன்", "பிறகு", "அவர்கள்", "வரை", "அவள்", "நீ", "ஆகிய", "இருந்தது", "உள்ளன",
    "வந்த", "இருந்த", "மிகவும்", "இங்கு", "மீது", "ஓர்", "இவை", "இந்தக்", "பற்றி", "வரும்",
    "வேறு", "இரு", "இதில்", "போல்", "இப்போது", "அவரது", "மட்டும்", "இந்தப்", "எனும்", "மேல்",
    "பின்", "சேர்ந்த", "ஆகியோர்", "எனக்கு", "இன்னும்", "அந்தப்", "அன்று", "ஒரே", "மிக", "அங்கு",
    "பல்வேறு", "விட்டு", "பெரும்", "அதை", "பற்றிய", "உன்", "அதிக", "அந்தக்", "பேர்", "இதனால்",
    "அவை", "அதே", "ஏன்", "முறை", "யார்", "என்பதை", "எல்லாம்", "மட்டுமே", "இங்கே", "அங்கே",
    "இடம்", "இடத்தில்", "அதில்", "நாம்", "அதற்கு", "எனவே", "பிற", "சிறு", "மற்ற", "விட", "எந்த",
    "எனவும்", "எனப்படும்", "எனினும்", "அடுத்த", "இதனை", "இதை", "கொள்ள", "இந்தத்", "இதற்கு",
    "அதனால்", "தவிர", "போல", "வரையில்", "சற்று", "எனக்"
    ]

    text_ls = text.split()
    filtered_words = [word for word in text_ls if word not in tamil_stopwords]
    # Join the remaining words back into a string
    text = " ".join(filtered_words)

    from indic_transliteration import sanscript
    from indic_transliteration.sanscript import SchemeMap, SCHEMES, transliterate
    
    
    text = transliterate(text, sanscript.HK, sanscript.TAMIL)


    
    return text

In [ ]:
train['transcriptions'] = train['transcriptions'].apply(text_preprocessing)
test['transcriptions'] = test['transcriptions'].apply(text_preprocessing)
dev['transcriptions'] = dev['transcriptions'].apply(text_preprocessing)

In [ ]:
# !pip install googletrans==4.0.0-rc1

In [ ]:
pip install deep-translator


In [ ]:
from PIL import Image
from torchvision import transforms
from deep_translator import GoogleTranslator
import numpy as np
import pandas as pd

augmented_data = []
cnt = 0
img_augmentations = transforms.Compose([
    transforms.ColorJitter(brightness=0.5),
    transforms.RandomGrayscale(p=0.2),
    transforms.RandomPosterize(bits=4),
    # transforms.GaussianBlur(kernel_size=3)
])

for idx, row in train.iterrows():
    image_data = row['image_data']
    text = row['transcriptions']
    label = row['labels']
    image_id = row['image_id']

    if label == 1:
        cnt += 1
        print(cnt)


        image = Image.fromarray(image_data.astype('uint8'))
        img_aug = img_augmentations(image)
        img_aug = np.array(img_aug)


        text_aug = text
        try:
            translated_to_en = GoogleTranslator(source='ta', target='en').translate(text_aug)
            text_aug = GoogleTranslator(source='en', target='ta').translate(translated_to_en)
        except Exception as e:
            print(f"Translation error (Tamil -> English -> Tamil): {e}")
            print(text)
        
        augmented_data.append({
            'image_data': img_aug,
            'labels': label,
            'image_id': image_id,
            'transcriptions': text_aug
        })

        image = Image.fromarray(image_data.astype('uint8'))
        img_aug = img_augmentations(image)
        img_aug = np.array(img_aug)


        text_aug = text
        try:
            translated_to_ml = GoogleTranslator(source='ta', target='ml').translate(text_aug)
            text_aug = GoogleTranslator(source='ml', target='ta').translate(translated_to_ml)
        except Exception as e:
            print(f"Translation error (Tamil -> Malayalam -> Tamil): {e}")
            print(text)
        
        augmented_data.append({
            'image_data': img_aug,
            'labels': label,
            'image_id': image_id,
            'transcriptions': text_aug
        })


augmented_df = pd.DataFrame(augmented_data)


train = pd.concat([train, augmented_df], ignore_index=True)


In [ ]:
train.drop(columns='image_id', inplace=True)
dev.drop(columns='image_id', inplace=True)
test.drop(columns='image_id', inplace=True)

# mBert + ViT

In [ ]:
import torch
from transformers import BertTokenizer, BertModel, ViTModel, ViTFeatureExtractor
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from PIL import Image


class MultiModalDataset(Dataset):
    def __init__(self, df, tokenizer, feature_extractor):
        self.df = df
        self.tokenizer = tokenizer
        self.feature_extractor = feature_extractor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, 'transcriptions']
        label = self.df.loc[idx, 'labels']
        image = self.df.loc[idx, 'image_data'] 

       
        encoding = self.tokenizer(text, padding='max_length', truncation=True, max_length=512, return_tensors='pt')

       
        image = Image.fromarray(image)
        image = self.feature_extractor(images=image, return_tensors='pt')['pixel_values'].squeeze(0)

        return encoding, image, torch.tensor(label)


tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
bert_model = BertModel.from_pretrained('bert-base-multilingual-cased')


feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224')
vit_model = ViTModel.from_pretrained('google/vit-base-patch16-224')


class MultiModal(nn.Module):
    def __init__(self, bert_model, vit_model, num_classes):
        super(MultiModal, self).__init__()
        self.bert_model = bert_model
        self.vit_model = vit_model
        self.fc = nn.Linear(768 + 768, num_classes)

    def forward(self, input_ids, attention_mask, pixel_values):

        bert_output = self.bert_model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        bert_pool = torch.mean(bert_output, 1)

 
        vit_output = self.vit_model(pixel_values=pixel_values).last_hidden_state
        vit_pool = torch.mean(vit_output, 1)

  
        combined_features = torch.cat((bert_pool, vit_pool), dim=1)
        output = self.fc(combined_features)
        
        return output


train_dataset = MultiModalDataset(train, tokenizer, feature_extractor)
dev_dataset = MultiModalDataset(dev, tokenizer, feature_extractor)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=16)


num_classes = len(train['labels'].unique())
model = MultiModal(bert_model, vit_model, num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

for epoch in range(20):  
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        input_ids = batch[0]['input_ids'].squeeze(1).to(device)
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)
        pixel_values = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask, pixel_values)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

# Evaluation
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for batch in dev_loader:
        input_ids = batch[0]['input_ids'].squeeze(1).to(device)
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)
        pixel_values = batch[1].to(device)
        labels = batch[2].to(device)

        outputs = model(input_ids, attention_mask, pixel_values)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Accuracy: {100 * correct / total}%")


In [ ]:
from sklearn.metrics import classification_report, f1_score


model.eval()
with torch.no_grad():
    y_true = []
    y_pred = []
    
    for batch in dev_loader:
        input_ids = batch[0]['input_ids'].squeeze(1).to(device)
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)
        images = batch[1].to(device)
        labels = batch[2].to(device)

        outputs = model(input_ids, attention_mask, images)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

    accuracy = 100 * sum(np.array(y_true) == np.array(y_pred)) / len(y_true)
    f1 = f1_score(y_true, y_pred, average='macro')
    class_report = classification_report(y_true, y_pred)

    print(f"Accuracy: {accuracy}%")
    print(f"F1-score: {f1}")
    print("Classification Report:")
    print(class_report)


In [ ]:
df_test = load_images_to_dataframe('/kaggle/input/tamillang/Misogyny Meme Detection/test')
test_csv = pd.read_csv('/kaggle/input/tamillang/Misogyny Meme Detection/test/test.csv')
test = pd.merge(test_csv, df_test, on='image_id', how='inner')

test['transcriptions'] = test['transcriptions'].apply(text_preprocessing)
test

In [ ]:
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import pandas as pd
import torch
import numpy as np


class TestDataset(Dataset):
    def __init__(self, df, tokenizer, feature_extractor):
        self.df = df
        self.tokenizer = tokenizer
        self.feature_extractor = feature_extractor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, 'transcriptions']
        image = self.df.loc[idx, 'image_data'] 

   
        encoding = self.tokenizer(text, padding='max_length', truncation=True, max_length=512, return_tensors='pt')

   
        image = Image.fromarray(image)
        image = self.feature_extractor(images=image, return_tensors='pt')['pixel_values'].squeeze(0)

        return encoding, image


test_dataset = TestDataset(test, tokenizer, feature_extractor)
test_loader = DataLoader(test_dataset, batch_size=16)


model.eval()


test_predictions = []

with torch.no_grad():
    for batch in test_loader:
        # Unpack batch data
        input_ids = batch[0]['input_ids'].squeeze(1).to(device)
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)
        pixel_values = batch[1].to(device)

     
        outputs = model(input_ids, attention_mask, pixel_values)
        _, predicted = torch.max(outputs, 1)

     
        test_predictions.extend(predicted.cpu().numpy())


test_predictions = np.array(test_predictions)


test['predictions'] = test_predictions


predictions_df = pd.DataFrame({
    'image_id': test['image_id'],
    'predictions': test_predictions
})

predictions_df.to_csv('mBert+ViT Prediction.csv', index=False)
print("Predictions saved")


# mBert+EfficientNet

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from PIL import Image

class MultiModalDataset(Dataset):
    def __init__(self, df, tokenizer, image_transform):
        self.df = df
        self.tokenizer = tokenizer
        self.image_transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, 'transcriptions']
        label = self.df.loc[idx, 'labels']
        image = self.df.loc[idx, 'image_data'] 
        
        encoding = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=128, return_tensors='pt'
        )

        
        image = Image.fromarray(image)
        image = self.image_transform(image)

        return encoding, image, torch.tensor(label)


tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
mbert_model = AutoModel.from_pretrained('bert-base-multilingual-cased')


efficientnet_model = models.efficientnet_b0(pretrained=True)
efficientnet_model.classifier = nn.Identity()  


image_transform = transforms.Compose([
    transforms.Resize((224, 224)),  
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class MultiModal(nn.Module):
    def __init__(self, mbert_model, efficientnet_model, num_classes):
        super(MultiModal, self).__init__()
        self.mbert_model = mbert_model
        self.efficientnet_model = efficientnet_model
        self.fc = nn.Linear(768 + 1280, num_classes)  # mBERT: 768-d, EfficientNet-B0: 1280-d

    def forward(self, input_ids, attention_mask, pixel_values):
        # Text embeddings from mBERT
        mbert_output = self.mbert_model(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state
        mbert_pool = torch.mean(mbert_output, 1)

        efficientnet_features = self.efficientnet_model(pixel_values)


        combined_features = torch.cat((mbert_pool, efficientnet_features), dim=1)
        output = self.fc(combined_features)
        return output


train_dataset = MultiModalDataset(train, tokenizer, image_transform)
dev_dataset = MultiModalDataset(dev, tokenizer, image_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=16)


num_classes = len(train['labels'].unique())
model = MultiModal(mbert_model, efficientnet_model, num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

for epoch in range(20):  
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        input_ids = batch[0]['input_ids'].squeeze(1).to(device)
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)
        pixel_values = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask, pixel_values)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

# Evaluation
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for batch in dev_loader:
        input_ids = batch[0]['input_ids'].squeeze(1).to(device)
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)
        pixel_values = batch[1].to(device)
        labels = batch[2].to(device)

        outputs = model(input_ids, attention_mask, pixel_values)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Accuracy: {100 * correct / total}%")


In [ ]:
from sklearn.metrics import classification_report, f1_score
import numpy as np


model.eval()
with torch.no_grad():
    y_true = []
    y_pred = []

    for batch in dev_loader:

        input_ids = batch[0]['input_ids'].squeeze(1).to(device)  
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)
        pixel_values = batch[1].to(device)  
        labels = batch[2].to(device)  


        outputs = model(input_ids, attention_mask, pixel_values) 
        _, predicted = torch.max(outputs, 1)  


        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())


accuracy = 100 * (np.array(y_true) == np.array(y_pred)).sum() / len(y_true)
f1 = f1_score(y_true, y_pred, average='macro')  
class_report = classification_report(y_true, y_pred, digits=4) 

# Print metrics
print(f"Accuracy: {accuracy:.2f}%")
print(f"F1-score (macro): {f1:.4f}")
print("Classification Report:")
print(class_report)


In [ ]:
df_test = load_images_to_dataframe('/kaggle/input/tamillang/Misogyny Meme Detection/test')
test_csv = pd.read_csv('/kaggle/input/tamillang/Misogyny Meme Detection/test/test.csv')
test = pd.merge(test_csv, df_test, on='image_id', how='inner')

test['transcriptions'] = test['transcriptions'].apply(text_preprocessing)
test

In [ ]:
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import pandas as pd
import torch
import numpy as np


class TestDataset(Dataset):
    def __init__(self, df, tokenizer, image_transform):
        self.df = df
        self.tokenizer = tokenizer
        self.image_transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, 'transcriptions']
        image = self.df.loc[idx, 'image_data']  


        encoding = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=512, return_tensors='pt'
        )


        image = Image.fromarray(image)
        image = self.image_transform(image)

        return encoding, image

test_dataset = TestDataset(test, tokenizer, image_transform)
test_loader = DataLoader(test_dataset, batch_size=16)

model.eval()

test_predictions = []

with torch.no_grad():
    for batch in test_loader:

        input_ids = batch[0]['input_ids'].squeeze(1).to(device)
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)
        pixel_values = batch[1].to(device)

        outputs = model(input_ids, attention_mask, pixel_values)
        _, predicted = torch.max(outputs, 1)


        test_predictions.extend(predicted.cpu().numpy())


test_predictions = np.array(test_predictions)


test['predictions'] = test_predictions


predictions_df = pd.DataFrame({
    'image_id': test['image_id'],
    'predictions': test_predictions
})

# Save to CSV
predictions_df.to_csv('mBert+Efficient.csv', index=False)
print("Predictions saved")


# Indic + Resnet

In [ ]:
from transformers import AutoTokenizer, AutoModel
from torchvision import models, transforms
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from PIL import Image


class MultiModalDataset(Dataset):
    def __init__(self, df, tokenizer, image_transform):
        self.df = df
        self.tokenizer = tokenizer
        self.image_transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, 'transcriptions']
        label = self.df.loc[idx, 'labels']
        image = self.df.loc[idx, 'image_data'] 


        encoding = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=128, return_tensors='pt'
        )


        image = Image.fromarray(image)
        image = self.image_transform(image)

        return encoding, image, torch.tensor(label)


tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-bert')
indicbert_model = AutoModel.from_pretrained('ai4bharat/indic-bert')


resnet_model = models.resnet50(pretrained=True)
resnet_model.fc = nn.Identity()  # Remove classification head


image_transform = transforms.Compose([
    transforms.Resize((224, 224)),  
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class MultiModal(nn.Module):
    def __init__(self, indicbert_model, resnet_model, num_classes):
        super(MultiModal, self).__init__()
        self.indicbert_model = indicbert_model
        self.resnet_model = resnet_model
        self.fc = nn.Linear(768 + 2048, num_classes) 

    def forward(self, input_ids, attention_mask, pixel_values):

        indicbert_output = self.indicbert_model(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state
        indicbert_pool = torch.mean(indicbert_output, 1)


        resnet_features = self.resnet_model(pixel_values)


        combined_features = torch.cat((indicbert_pool, resnet_features), dim=1)
        output = self.fc(combined_features)
        return output


train_dataset = MultiModalDataset(train, tokenizer, image_transform)
dev_dataset = MultiModalDataset(dev, tokenizer, image_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=16)


num_classes = len(train['labels'].unique())
model = MultiModal(indicbert_model, resnet_model, num_classes)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

for epoch in range(20):  
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        input_ids = batch[0]['input_ids'].squeeze(1).to(device)
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)
        pixel_values = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask, pixel_values)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

# Evaluation
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for batch in dev_loader:
        input_ids = batch[0]['input_ids'].squeeze(1).to(device)
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)
        pixel_values = batch[1].to(device)
        labels = batch[2].to(device)

        outputs = model(input_ids, attention_mask, pixel_values)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Accuracy: {100 * correct / total}%")

In [ ]:
from sklearn.metrics import classification_report, f1_score
import numpy as np


model.eval()
with torch.no_grad():
    y_true = []
    y_pred = []

    for batch in dev_loader:
        input_ids = batch[0]['input_ids'].squeeze(1).to(device)  
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)  
        pixel_values = batch[1].to(device)  
        labels = batch[2].to(device)  

        # Get model predictions
        outputs = model(input_ids, attention_mask, pixel_values)  
        _, predicted = torch.max(outputs, 1) 


        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())


accuracy = 100 * (np.array(y_true) == np.array(y_pred)).sum() / len(y_true)
f1 = f1_score(y_true, y_pred, average='macro') 
class_report = classification_report(y_true, y_pred, digits=4)  

# Print metrics
print(f"Accuracy: {accuracy:.2f}%")
print(f"F1-score (macro): {f1:.4f}")
print("Classification Report:")
print(class_report)


In [ ]:
df_test = load_images_to_dataframe('/kaggle/input/tamillang/Misogyny Meme Detection/test')
test_csv = pd.read_csv('/kaggle/input/tamillang/Misogyny Meme Detection/test/test.csv')
test = pd.merge(test_csv, df_test, on='image_id', how='inner')

test['transcriptions'] = test['transcriptions'].apply(text_preprocessing)
test

In [ ]:
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import pandas as pd
import torch
import numpy as np


class TestDataset(Dataset):
    def __init__(self, df, tokenizer, image_transform):
        self.df = df
        self.tokenizer = tokenizer
        self.image_transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, 'transcriptions']
        image = self.df.loc[idx, 'image_data']  


        encoding = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=512, return_tensors='pt'
        )


        image = Image.fromarray(image)
        image = self.image_transform(image)

        return encoding, image


test_dataset = TestDataset(test, tokenizer, image_transform)
test_loader = DataLoader(test_dataset, batch_size=16)


model.eval()

# Store predictions
test_predictions = []

with torch.no_grad():
    for batch in test_loader:
        # Unpack batch data
        input_ids = batch[0]['input_ids'].squeeze(1).to(device)
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)
        pixel_values = batch[1].to(device)

        outputs = model(input_ids, attention_mask, pixel_values)
        _, predicted = torch.max(outputs, 1)


        test_predictions.extend(predicted.cpu().numpy())


test_predictions = np.array(test_predictions)


test['predictions'] = test_predictions


predictions_df = pd.DataFrame({
    'image_id': test['image_id'],
    'predictions': test_predictions
})

# Save to CSV
predictions_df.to_csv('Indic+Resnet.csv', index=False)
print("Predictions saved")
